# QPU 실험: Hardened Posiform α=0

## 실험 목적

Hardened Posiform에서 α=0인 경우 `Q = Σ R_i` (순수 block-diagonal random QUBO).
SA 실험에서 **에너지 성공률 89.8%, 비트 성공률 0%** (축퇴 때문)였던 결과를 D-Wave QPU에서 재현한다.

**핵심 질문:**
1. QPU도 SA처럼 GS 에너지는 쉽게 찾되, planted target은 못 찾는가?
2. QPU solution의 Hamming distance 분포가 SA와 다른가?
3. 하드웨어 노이즈가 축퇴를 깨는 효과가 있는가?

## α=0에서의 수학적 배경

```
Q(x) = R(x) + α·P(x)  →  α=0이면  Q(x) = R(x)
```

- R은 block-diagonal (k-bit 독립 subproblem의 합)
- target x*는 각 subproblem의 GS를 concatenate → R의 GS
- 축퇴도 |GS(R)| = Π_i |GS(R_i)| (기하급수적)
- SA/QPU 모두 GS **에너지**는 쉽게 찾지만, 특정 target **비트스트링**은 찾지 못함

## 1. D-Wave Leap API 토큰 설정

### 토큰 확인
1. https://cloud.dwavesys.com 로그인
2. 좌측 메뉴 **API Tokens** 클릭
3. 토큰 복사 (예: `DEV-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx`)

### 설정 방법 (아래 셋 중 하나 선택)

**방법 1: 노트북에서 직접 (가장 간단)**
```python
# 아래 코드 셀에서 token= 파라미터에 직접 입력
qpu = DWaveSampler(token="DEV-여기에토큰붙여넣기")
```

**방법 2: 환경변수**
```bash
# 터미널에서 실행 후 Jupyter 재시작
export DWAVE_API_TOKEN="DEV-여기에토큰붙여넣기"
```

**방법 3: dwave CLI 설정 (영구 저장)**
```bash
# 터미널에서 실행
dwave config create
# Profile: defaults
# API endpoint URL: https://cloud.dwavesys.com/sapi/
# Authentication token: DEV-여기에토큰붙여넣기
# Default client: auto
# Default solver: (빈칸 Enter)
```

### Leap 대시보드에서 확인할 것
- **Solver access**: Advantage_system (Pegasus) 또는 Advantage2_prototype (Zephyr)
- **QPU time remaining**: 무료 플랜은 월 1분. 이 노트북의 기본 설정은 ~10초 QPU 시간 사용

In [13]:
import sys
print(f"Python: {sys.executable}")
!{sys.executable} -m pip install dwave-system dwave-neal numpy matplotlib

Python: /home/yideun/anaconda3/bin/bin/python


In [14]:
from dwave.system import DWaveSampler, EmbeddingComposite
import neal
import numpy as np
from itertools import product
import random
import time
import json
import os

np.set_printoptions(threshold=200, linewidth=150, precision=3, suppress=True)

# ═══════════════════════════════════════════════════════
# 토큰 설정: 아래 두 방법 중 하나 선택
# ═══════════════════════════════════════════════════════

# 방법 1: 직접 입력 (토큰을 여기에 붙여넣기)
MY_TOKEN = None  # ← "DEV-xxxxx" 형태로 변경

# 방법 2: 환경변수 또는 dwave config에 이미 설정된 경우 None 유지
# ═══════════════════════════════════════════════════════

if MY_TOKEN:
    qpu = DWaveSampler(token=MY_TOKEN)
else:
    qpu = DWaveSampler(token="yXwP-644aea66ae6f2920550b5c28a53d47e018fdf32e")  # 환경변수 또는 ~/.config/dwave/dwave.conf 사용

print(f"QPU: {qpu.solver.name}")
print(f"Qubits: {qpu.solver.num_qubits}")
print(f"Topology: {qpu.properties.get('topology', {}).get('type', 'unknown')}")

qpu_sampler = EmbeddingComposite(qpu)
sa_sampler = neal.SimulatedAnnealingSampler()

print("\nD-Wave 연결 성공!")

QPU: Advantage_system4.1
Qubits: 5760
Topology: pegasus

D-Wave 연결 성공!


## 2. QUBO 생성 함수 (α=0: 순수 block-diagonal random QUBO)

In [15]:
COEFF_LIN2 = [-1, 1]
COEFF_LIN20 = [round(-1 + 0.1 * i, 1) for i in range(21)]


def gen_random_qubo(n, coeff_type='lin2'):
    """k-bit random discrete-coefficient QUBO (상삼각)."""
    coeffs = COEFF_LIN2 if coeff_type == 'lin2' else COEFF_LIN20
    mat = np.array([[random.choice(coeffs) for _ in range(n)] for _ in range(n)])
    return np.triu(mat)


def find_opt_brute_force(mat):
    """brute force GS 탐색. Returns (best_x, best_val, num_degenerate)."""
    n = mat.shape[0]
    best_x, best_val, num_degenerate = None, float('inf'), 0

    for bits in product([0, 1], repeat=n):
        x = np.array(bits)
        val = x @ mat @ x
        if val < best_val - 1e-12:
            best_val, best_x, num_degenerate = val, x, 1
        elif abs(val - best_val) < 1e-12:
            num_degenerate += 1

    return best_x, best_val, num_degenerate


def gen_block_diagonal_qubo(n, max_sub_graph_size, coeff_type='lin2'):
    """
    α=0 Hardened Posiform = block-diagonal random QUBO.
    각 subgraph의 GS를 brute force로 계산하여 planted target 구성.

    Returns:
        Q_dict: {(i,j): weight} — D-Wave/neal 공용 입력 형식
        target: planted target bitstring
        info: 메타정보 (subblock 축퇴도, 에너지 등)
    """
    k = max(1, -(-n // max_sub_graph_size))  # ceil division
    Q_dict = {}
    target_bits = np.zeros(n, dtype=int)
    total_energy = 0.0
    total_degenerate = 1
    block_info = []

    for b in range(k):
        start = b * n // k
        end = (b + 1) * n // k
        size = end - start
        variables = list(range(start, end))

        # subblock random QUBO 생성
        sub_mat = gen_random_qubo(size, coeff_type)
        sub_opt, sub_energy, sub_deg = find_opt_brute_force(sub_mat)

        # target에 기록
        target_bits[start:end] = sub_opt
        total_energy += sub_energy
        total_degenerate *= sub_deg

        # Q_dict에 변환 (global 인덱스)
        for i in range(size):
            for j in range(i, size):
                if sub_mat[i][j] != 0:
                    Q_dict[(variables[i], variables[j])] = float(sub_mat[i][j])

        block_info.append({'start': start, 'end': end, 'size': size,
                           'energy': sub_energy, 'degeneracy': sub_deg})

    target = ''.join(map(str, target_bits))
    info = {
        'n': n, 'num_blocks': k, 'coeff_type': coeff_type,
        'target': target, 'target_energy': total_energy,
        'total_degeneracy': total_degenerate,
        'blocks': block_info,
    }
    return Q_dict, target, info


print("함수 정의 완료")

함수 정의 완료


In [16]:
def analyze_samples(response, target, n):
    """
    SampleSet 분석.
    Returns dict: eng_ok, bit_ok, hd_list, best_energy, num_samples
    """
    target_energy_computed = None
    eng_ok = 0
    bit_ok = 0
    hd_list = []
    best_energy = float('inf')
    energy_tol = 1e-2  # QPU는 아날로그 노이즈 → SA보다 넓은 tolerance

    for sample, energy, num_occ in response.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(n))
        hd = sum(1 for a, b in zip(target, found) if a != b)
        hd_list.append(hd)
        best_energy = min(best_energy, energy)
        if found == target:
            bit_ok += 1
        # target 에너지 계산 (첫 번째만)
        if target_energy_computed is None:
            # response에서 target의 에너지를 직접 계산
            target_energy_computed = sum(
                w * sample.get(i, 0) * (sample.get(j, 0) if i != j else 1)
                for (i, j), w in response.vartype is not None and []
            )

    # target_energy는 info에서 가져와야 하므로 외부에서 전달
    return {
        'eng_ok': eng_ok,
        'bit_ok': bit_ok,
        'hd_list': hd_list,
        'hd_avg': np.mean(hd_list) if hd_list else 0,
        'hd_med': np.median(hd_list) if hd_list else 0,
        'best_energy': best_energy,
        'num_samples': len(hd_list),
    }


def analyze_with_energy(response, target, target_energy, n, energy_tol=1e-2):
    """에너지 비교 포함 분석."""
    eng_ok = 0
    bit_ok = 0
    gs_not_target = 0  # GS 에너지 도달했지만 target이 아닌 경우
    hd_list = []
    best_energy = float('inf')

    for sample, energy, num_occ in response.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(n))
        hd = sum(1 for a, b in zip(target, found) if a != b)
        hd_list.append(hd)
        best_energy = min(best_energy, energy)

        is_gs = abs(energy - target_energy) < energy_tol
        is_target = (found == target)

        if is_gs:
            eng_ok += 1
        if is_target:
            bit_ok += 1
        if is_gs and not is_target:
            gs_not_target += 1

    return {
        'eng_ok': eng_ok, 'bit_ok': bit_ok,
        'gs_not_target': gs_not_target,
        'hd_list': hd_list,
        'hd_avg': np.mean(hd_list) if hd_list else 0,
        'hd_med': int(np.median(hd_list)) if hd_list else 0,
        'best_energy': best_energy,
        'num_samples': len(hd_list),
    }


print("분석 함수 정의 완료")

분석 함수 정의 완료


## 3. 단일 인스턴스 테스트 (QPU 동작 확인)

QPU 연결 + embedding이 정상 동작하는지 작은 인스턴스로 확인.

In [18]:
# ─── 작은 인스턴스로 QPU 테스트 ───
random.seed(42)
Q_test, target_test, info_test = gen_block_diagonal_qubo(
    n=500, max_sub_graph_size=10, coeff_type='lin2'
)

print(f"[QUBO 생성]")
print(f"  n={info_test['n']}, blocks={info_test['num_blocks']}")
print(f"  target: {target_test}")
print(f"  target energy: {info_test['target_energy']:.1f}")
print(f"  total degeneracy: {info_test['total_degeneracy']}")
for i, b in enumerate(info_test['blocks']):
    print(f"  block {i}: vars [{b['start']},{b['end']}), deg={b['degeneracy']}, E={b['energy']:.1f}")

# QPU 실행
print(f"\n[QPU 실행] num_reads=100")
t0 = time.time()
qpu_response = qpu_sampler.sample_qubo(Q_test, num_reads=100, annealing_time=20)
qpu_time = time.time() - t0

qpu_result = analyze_with_energy(
    qpu_response, target_test, info_test['target_energy'],
    info_test['n'], energy_tol=0.5  # QPU는 아날로그 노이즈가 있으므로 tolerance 넓게
)

print(f"  QPU wall time: {qpu_time:.2f}s")
print(f"  QPU timing: {qpu_response.info.get('timing', {})}")
print(f"  Best energy: {qpu_result['best_energy']:.3f} (target: {info_test['target_energy']:.1f})")
print(f"  Energy 성공: {qpu_result['eng_ok']}/{qpu_result['num_samples']}")
print(f"  Bit 성공:    {qpu_result['bit_ok']}/{qpu_result['num_samples']}")
print(f"  GS≠Target:  {qpu_result['gs_not_target']}")
print(f"  Avg HD:      {qpu_result['hd_avg']:.1f}")

# SA 비교
print(f"\n[SA 실행] num_reads=100, num_sweeps=1000")
sa_response = sa_sampler.sample_qubo(Q_test, num_reads=100, num_sweeps=1000)
sa_result = analyze_with_energy(
    sa_response, target_test, info_test['target_energy'],
    info_test['n'], energy_tol=1e-6
)
print(f"  Best energy: {sa_result['best_energy']:.3f}")
print(f"  Energy 성공: {sa_result['eng_ok']}/{sa_result['num_samples']}")
print(f"  Bit 성공:    {sa_result['bit_ok']}/{sa_result['num_samples']}")
print(f"  GS≠Target:  {sa_result['gs_not_target']}")
print(f"  Avg HD:      {sa_result['hd_avg']:.1f}")

[QUBO 생성]
  n=500, blocks=50
  target: 11011101010110110011101111011001110000111010011111011111011000110101111011010000001101100001100111001101111001111101100101111110011100100011111011011000011111110000111011001110001111010100010110011110101101010011111010010100100101100010101110011111010110111101000111110010001101011101101100111011111111110100001010101101010100011001001101011111111000100110011110011101011111011100001110100111111110101111010011011110001000101100110110111011110010011101100111001000010111101010111001011111001111100111
  target energy: -472.0
  total degeneracy: 17199267840
  block 0: vars [0,10), deg=1, E=-16.0
  block 1: vars [10,20), deg=1, E=-11.0
  block 2: vars [20,30), deg=1, E=-12.0
  block 3: vars [30,40), deg=2, E=-11.0
  block 4: vars [40,50), deg=1, E=-10.0
  block 5: vars [50,60), deg=2, E=-14.0
  block 6: vars [60,70), deg=1, E=-7.0
  block 7: vars [70,80), deg=1, E=-4.0
  block 8: vars [80,90), deg=1, E=-6.0
  block 9: vars [90,100), deg=6, E=-5.0
  block 

## 4. 본 실험: QPU vs SA 비교 (α=0)

### 실험 설계

동일 QUBO 인스턴스에 대해 QPU와 SA를 모두 실행하여 직접 비교.

**측정 지표:**
- **Energy 성공률**: `|best_energy - target_energy| < tol` 인 샘플 비율
- **Bit 성공률**: target bitstring과 정확히 일치하는 샘플 비율
- **GS≠Target**: GS 에너지에 도달했지만 target이 아닌 샘플 수 (축퇴 효과)
- **Hamming Distance**: SA/QPU 솔루션과 planted target의 HD 분포

In [ ]:
# ═══════════════════════════════════════════
# 하이퍼파라미터 — 필요에 따라 조정
# ═══════════════════════════════════════════
N = 100                   # 변수 수
MAX_SUB = 10              # subgraph 크기 (block 당 변수 수)
COEFF_TYPE = 'lin2'       # 'lin2' ({-1,+1}) or 'lin20'
NUM_INSTANCES = 50        # 인스턴스 수
QPU_NUM_READS = 100       # QPU 샘플 수/인스턴스
QPU_ANNEALING_TIME = 20   # 어닐링 시간 (μs). D-Wave 기본값 20
SA_NUM_READS = 100        # SA 샘플 수/인스턴스
SA_NUM_SWEEPS = 1000      # SA sweep 수
QPU_ENERGY_TOL = 0.5      # QPU 에너지 tolerance (아날로그 노이즈)
SA_ENERGY_TOL = 1e-6      # SA 에너지 tolerance

print(f"═══ QPU vs SA 실험 (α=0) ═══")
print(f"  N={N}, block_size={MAX_SUB}, coeff={COEFF_TYPE}")
print(f"  instances={NUM_INSTANCES}")
print(f"  QPU: reads={QPU_NUM_READS}, annealing={QPU_ANNEALING_TIME}μs, tol={QPU_ENERGY_TOL}")
print(f"  SA:  reads={SA_NUM_READS}, sweeps={SA_NUM_SWEEPS}, tol={SA_ENERGY_TOL}")

In [ ]:
# ─── 실험 루프 ───
results_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                           'hardened_posiform', 'results')
os.makedirs(results_dir, exist_ok=True)
results_path = os.path.join(results_dir,
    f'qpu_alpha0_n{N}_sub{MAX_SUB}_{COEFF_TYPE}_inst{NUM_INSTANCES}.json')

# 누적 통계
qpu_stats = {'eng_ok': 0, 'bit_ok': 0, 'gs_not_target': 0, 'total': 0, 'hd_sum': 0.0}
sa_stats  = {'eng_ok': 0, 'bit_ok': 0, 'gs_not_target': 0, 'total': 0, 'hd_sum': 0.0}
all_results = []

t0 = time.perf_counter()

for inst in range(NUM_INSTANCES):
    random.seed(inst * 53)

    # QUBO 생성
    Q, target, info = gen_block_diagonal_qubo(N, MAX_SUB, COEFF_TYPE)
    target_energy = info['target_energy']
    n = info['n']

    # ── QPU ──
    try:
        qpu_resp = qpu_sampler.sample_qubo(
            Q, num_reads=QPU_NUM_READS, annealing_time=QPU_ANNEALING_TIME
        )
        qpu_r = analyze_with_energy(qpu_resp, target, target_energy, n, QPU_ENERGY_TOL)
        qpu_timing = qpu_resp.info.get('timing', {})
    except Exception as e:
        print(f"  [inst {inst}] QPU error: {e}")
        qpu_r = {'eng_ok': 0, 'bit_ok': 0, 'gs_not_target': 0,
                  'hd_avg': n/2, 'hd_med': n//2, 'best_energy': float('inf'),
                  'num_samples': 0, 'hd_list': []}
        qpu_timing = {}

    # ── SA ──
    sa_resp = sa_sampler.sample_qubo(Q, num_reads=SA_NUM_READS, num_sweeps=SA_NUM_SWEEPS)
    sa_r = analyze_with_energy(sa_resp, target, target_energy, n, SA_ENERGY_TOL)

    # 누적
    for stats, r in [(qpu_stats, qpu_r), (sa_stats, sa_r)]:
        stats['eng_ok'] += r['eng_ok']
        stats['bit_ok'] += r['bit_ok']
        stats['gs_not_target'] += r['gs_not_target']
        stats['total'] += r['num_samples']
        stats['hd_sum'] += sum(r['hd_list'])

    inst_data = {
        'inst': inst, 'target_energy': target_energy,
        'degeneracy': info['total_degeneracy'],
        'qpu_eng_ok': qpu_r['eng_ok'], 'qpu_bit_ok': qpu_r['bit_ok'],
        'qpu_gs_not_target': qpu_r['gs_not_target'],
        'qpu_hd_avg': round(qpu_r['hd_avg'], 2), 'qpu_hd_med': qpu_r['hd_med'],
        'qpu_best_E': qpu_r['best_energy'],
        'sa_eng_ok': sa_r['eng_ok'], 'sa_bit_ok': sa_r['bit_ok'],
        'sa_gs_not_target': sa_r['gs_not_target'],
        'sa_hd_avg': round(sa_r['hd_avg'], 2), 'sa_hd_med': sa_r['hd_med'],
        'sa_best_E': sa_r['best_energy'],
    }
    all_results.append(inst_data)

    # 진행 상황
    if (inst + 1) % 10 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (inst + 1) * (NUM_INSTANCES - inst - 1)
        qe = 100 * qpu_stats['eng_ok'] / max(1, qpu_stats['total'])
        se = 100 * sa_stats['eng_ok'] / max(1, sa_stats['total'])
        print(f"  [{inst+1:>3}/{NUM_INSTANCES}] {elapsed:>5.0f}s (ETA {eta:.0f}s) "
              f"QPU_eng:{qe:.1f}% SA_eng:{se:.1f}%")

elapsed = time.perf_counter() - t0
print(f"\n  완료: {elapsed:.1f}s")

# ─── 결과 저장 ───
with open(results_path, 'w') as f:
    json.dump({
        'params': {
            'n': N, 'max_sub': MAX_SUB, 'coeff': COEFF_TYPE,
            'num_instances': NUM_INSTANCES,
            'qpu_num_reads': QPU_NUM_READS, 'qpu_annealing_time': QPU_ANNEALING_TIME,
            'sa_num_reads': SA_NUM_READS, 'sa_num_sweeps': SA_NUM_SWEEPS,
            'qpu_energy_tol': QPU_ENERGY_TOL, 'sa_energy_tol': SA_ENERGY_TOL,
            'elapsed_s': round(elapsed, 1),
            'qpu_solver': qpu.solver.name,
        },
        'qpu_stats': qpu_stats,
        'sa_stats': sa_stats,
        'instances': all_results,
    }, f)

print(f"  저장: {results_path}")

## 5. 결과 요약

In [ ]:
# ─── 결과 테이블 ───
print(f"{'═' * 80}")
print(f"  QPU vs SA 비교 (α=0, N={N}, block={MAX_SUB}, {COEFF_TYPE})")
print(f"  {NUM_INSTANCES} instances × {QPU_NUM_READS} reads")
print(f"{'═' * 80}")
print(f"{'Solver':<8} {'Energy%':>10} {'Bit%':>10} {'GS≠Target':>12} {'Avg HD':>10}")
print(f"{'─' * 80}")

for name, stats, tol in [('QPU', qpu_stats, QPU_ENERGY_TOL),
                          ('SA', sa_stats, SA_ENERGY_TOL)]:
    total = max(1, stats['total'])
    eng_rate = 100 * stats['eng_ok'] / total
    bit_rate = 100 * stats['bit_ok'] / total
    avg_hd = stats['hd_sum'] / total
    print(f"{name:<8} {stats['eng_ok']:>5}/{total} ({eng_rate:>5.1f}%) "
          f"{stats['bit_ok']:>5}/{total} ({bit_rate:>5.1f}%) "
          f"{stats['gs_not_target']:>10} "
          f"{avg_hd:>9.1f}")

print(f"{'─' * 80}")
print(f"\n  QPU energy tol={QPU_ENERGY_TOL}, SA energy tol={SA_ENERGY_TOL}")

# 축퇴도 통계
degs = [r['degeneracy'] for r in all_results]
print(f"\n[축퇴도 통계]")
print(f"  min: {min(degs)}, max: {max(degs)}, median: {int(np.median(degs))}")
print(f"  평균 축퇴도 ≈ {np.mean(degs):.0f} (각 블록 축퇴도의 곱)")

## 6. Hamming Distance 분포 비교

In [ ]:
import matplotlib.pyplot as plt

# 인스턴스별 평균 HD 수집
qpu_hds = [r['qpu_hd_avg'] for r in all_results]
sa_hds = [r['sa_hd_avg'] for r in all_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) QPU vs SA: 인스턴스별 평균 HD scatter
ax = axes[0]
ax.scatter(sa_hds, qpu_hds, alpha=0.5, s=20)
max_hd = max(max(sa_hds, default=1), max(qpu_hds, default=1)) * 1.1
ax.plot([0, max_hd], [0, max_hd], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('SA Avg HD')
ax.set_ylabel('QPU Avg HD')
ax.set_title(f'(a) Instance-level HD (N={N}, α=0)')
ax.legend()
ax.set_aspect('equal')

# (b) HD 히스토그램 비교
ax = axes[1]
bins = np.arange(0, N//2 + 2, max(1, N//40))
ax.hist(qpu_hds, bins=bins, alpha=0.6, label='QPU', density=True)
ax.hist(sa_hds, bins=bins, alpha=0.6, label='SA', density=True)
ax.set_xlabel('Avg Hamming Distance')
ax.set_ylabel('Density')
ax.set_title(f'(b) HD Distribution')
ax.legend()

# (c) 인스턴스별 에너지 성공률
ax = axes[2]
qpu_eng_rates = [r['qpu_eng_ok'] / QPU_NUM_READS * 100 for r in all_results]
sa_eng_rates = [r['sa_eng_ok'] / SA_NUM_READS * 100 for r in all_results]
ax.scatter(sa_eng_rates, qpu_eng_rates, alpha=0.5, s=20)
ax.plot([0, 100], [0, 100], 'k--', alpha=0.3)
ax.set_xlabel('SA Energy Success %')
ax.set_ylabel('QPU Energy Success %')
ax.set_title(f'(c) Energy Success Rate')
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, f'qpu_alpha0_n{N}_{COEFF_TYPE}.png'), dpi=150)
plt.show()

print(f"\n[HD 요약]")
print(f"  QPU: avg={np.mean(qpu_hds):.1f}, median={np.median(qpu_hds):.0f}, "
      f"std={np.std(qpu_hds):.1f}")
print(f"  SA:  avg={np.mean(sa_hds):.1f}, median={np.median(sa_hds):.0f}, "
      f"std={np.std(sa_hds):.1f}")

## 7. 블록별 분석: QPU가 subblock GS를 개별적으로 찾는가?

α=0에서 Q는 block-diagonal. 각 블록을 독립적으로 풀 수 있으므로,
QPU 솔루션의 **블록별 정답률**을 확인한다.

SA 기존 실험에서 α=0의 energy 성공률 89.8%는 "대부분의 블록은 맞추되 1~2개 블록만 틀림"을 의미.

In [ ]:
# ─── 블록별 정답률 분석 (마지막 인스턴스 기준) ───
# 새 인스턴스 생성하여 블록별 상세 분석
random.seed(999)
Q_block, target_block, info_block = gen_block_diagonal_qubo(N, MAX_SUB, COEFF_TYPE)
n = info_block['n']
blocks = info_block['blocks']
num_blocks = info_block['num_blocks']

# QPU & SA 실행
qpu_resp_block = qpu_sampler.sample_qubo(Q_block, num_reads=200, annealing_time=QPU_ANNEALING_TIME)
sa_resp_block = sa_sampler.sample_qubo(Q_block, num_reads=200, num_sweeps=SA_NUM_SWEEPS)

def block_accuracy(response, target, blocks, n, num_reads):
    """각 블록별로 target과 일치하는 비율 계산."""
    block_correct = [0] * len(blocks)
    sample_count = 0
    for sample, energy, _ in response.data(['sample', 'energy', 'num_occurrences']):
        found = ''.join(str(sample[k]) for k in range(n))
        sample_count += 1
        for bi, b in enumerate(blocks):
            if found[b['start']:b['end']] == target[b['start']:b['end']]:
                block_correct[bi] += 1
    return [c / sample_count * 100 for c in block_correct]

qpu_block_acc = block_accuracy(qpu_resp_block, target_block, blocks, n, 200)
sa_block_acc = block_accuracy(sa_resp_block, target_block, blocks, n, 200)

print(f"[블록별 정답률] (N={N}, {num_blocks} blocks × {MAX_SUB} vars)")
print(f"{'Block':>6} {'Size':>5} {'Deg':>5} {'QPU%':>8} {'SA%':>8}")
print(f"{'─' * 40}")
for i, b in enumerate(blocks):
    print(f"{i:>6} {b['size']:>5} {b['degeneracy']:>5} "
          f"{qpu_block_acc[i]:>7.1f}% {sa_block_acc[i]:>7.1f}%")
print(f"{'─' * 40}")
print(f"{'Avg':>6} {'':>5} {'':>5} "
      f"{np.mean(qpu_block_acc):>7.1f}% {np.mean(sa_block_acc):>7.1f}%")

# 블록별 정답률 시각화
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(num_blocks)
width = 0.35
ax.bar(x - width/2, qpu_block_acc, width, label='QPU', alpha=0.7)
ax.bar(x + width/2, sa_block_acc, width, label='SA', alpha=0.7)
ax.set_xlabel('Block Index')
ax.set_ylabel('Block Accuracy (%)')
ax.set_title(f'Block-level Accuracy (α=0, N={N})')
ax.legend()
ax.set_xticks(x)
plt.tight_layout()
plt.show()

## 8. QPU Annealing Time 스윕 (선택)

어닐링 시간이 결과에 미치는 영향 확인. D-Wave 기본값은 20μs.

In [ ]:
# ─── Annealing Time 스윕 ───
anneal_times = [1, 5, 20, 100, 500, 2000]  # μs
anneal_num_reads = 200
anneal_num_instances = 20

anneal_stats = {t: {'eng_ok': 0, 'bit_ok': 0, 'gs_not_target': 0,
                     'total': 0, 'hd_sum': 0.0} for t in anneal_times}

print(f"═══ Annealing Time Sweep (α=0, N={N}) ═══")
print(f"  times(μs): {anneal_times}")
print(f"  instances={anneal_num_instances}, reads={anneal_num_reads}")

t0 = time.perf_counter()

for inst in range(anneal_num_instances):
    random.seed(inst * 77)
    Q, target, info = gen_block_diagonal_qubo(N, MAX_SUB, COEFF_TYPE)
    target_energy = info['target_energy']
    n = info['n']

    for at in anneal_times:
        try:
            resp = qpu_sampler.sample_qubo(
                Q, num_reads=anneal_num_reads, annealing_time=at
            )
            r = analyze_with_energy(resp, target, target_energy, n, QPU_ENERGY_TOL)
            anneal_stats[at]['eng_ok'] += r['eng_ok']
            anneal_stats[at]['bit_ok'] += r['bit_ok']
            anneal_stats[at]['gs_not_target'] += r['gs_not_target']
            anneal_stats[at]['total'] += r['num_samples']
            anneal_stats[at]['hd_sum'] += sum(r['hd_list'])
        except Exception as e:
            print(f"  [inst {inst}, at={at}μs] error: {e}")

    if (inst + 1) % 5 == 0:
        elapsed = time.perf_counter() - t0
        print(f"  [{inst+1}/{anneal_num_instances}] {elapsed:.0f}s")

# 결과 테이블
print(f"\n{'Anneal(μs)':<12} {'Energy%':>10} {'Bit%':>10} {'GS≠Target':>12} {'Avg HD':>10}")
print(f"{'─' * 60}")
for at in anneal_times:
    s = anneal_stats[at]
    total = max(1, s['total'])
    print(f"{at:<12} {s['eng_ok']:>5}/{total} ({100*s['eng_ok']/total:>5.1f}%) "
          f"{s['bit_ok']:>5}/{total} ({100*s['bit_ok']/total:>5.1f}%) "
          f"{s['gs_not_target']:>10} "
          f"{s['hd_sum']/total:>9.1f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

eng_rates = [100 * anneal_stats[t]['eng_ok'] / max(1, anneal_stats[t]['total'])
             for t in anneal_times]
hd_avgs = [anneal_stats[t]['hd_sum'] / max(1, anneal_stats[t]['total'])
           for t in anneal_times]

axes[0].semilogx(anneal_times, eng_rates, 'o-')
axes[0].set_xlabel('Annealing Time (μs)')
axes[0].set_ylabel('Energy Success Rate (%)')
axes[0].set_title('(a) Energy Success vs Annealing Time')
axes[0].set_ylim(-5, 105)

axes[1].semilogx(anneal_times, hd_avgs, 's-', color='tab:orange')
axes[1].set_xlabel('Annealing Time (μs)')
axes[1].set_ylabel('Avg Hamming Distance')
axes[1].set_title('(b) HD vs Annealing Time')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, f'qpu_anneal_sweep_n{N}_{COEFF_TYPE}.png'), dpi=150)
plt.show()

## 9. QPU Chain 정보 확인 (선택)

Embedding 품질 확인. Block-diagonal QUBO이므로 chain이 짧아야 정상.

In [ ]:
# ─── Embedding 정보 ───
# 마지막 QPU 실행의 embedding 정보 확인
embedding = qpu_resp_block.info.get('embedding_context', {}).get('embedding', {})
if embedding:
    chain_lengths = [len(chain) for chain in embedding.values()]
    print(f"[Embedding 정보]")
    print(f"  Logical qubits: {len(embedding)}")
    print(f"  Physical qubits: {sum(chain_lengths)}")
    print(f"  Chain lengths: min={min(chain_lengths)}, max={max(chain_lengths)}, "
          f"avg={np.mean(chain_lengths):.1f}")
    print(f"  Chain break fraction: "
          f"{qpu_resp_block.info.get('embedding_context', {}).get('chain_break_fraction', 'N/A')}")

    # chain length 분포
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(chain_lengths, bins=range(1, max(chain_lengths) + 2), alpha=0.7, edgecolor='black')
    ax.set_xlabel('Chain Length')
    ax.set_ylabel('Count')
    ax.set_title(f'Chain Length Distribution (N={N}, block-diagonal)')
    plt.tight_layout()
    plt.show()
else:
    print("Embedding 정보 없음 (simulator 사용 중?)")